Install all you need

In [17]:
import torch
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from transformers import AutoModelForSequenceClassification
import os
from scipy import stats
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import grangercausalitytests
import warnings 
from scipy.stats import pearsonr

You need to put the two datasets into ./data

output_dir is for checkpoints

file_path is for Embeddings in my Google Drive you can download: https://drive.google.com/file/d/1oDQXRH_38Vbli0vyyMdjZz_HwzRb1pOz/view?usp=sharing

one_minute_data_dir is for the Bitcoin data: https://www.kaggle.com/datasets/mczielinski/bitcoin-historical-data

In [ ]:
output_dir = r"./data/sentiment_batches/"
file_path = r"./data/tweets_2021_Embeddings.parquet"
one_minute_data_dir = r"./data/btcusd_1-min_data.csv"

Labeling the EMbeddings with 16gb ram for lower ram you need to change the code but i uploaded the results in Github so you can skip this code

In [2]:
robertamodel = AutoModelForSequenceClassification.from_pretrained(
    "cardiffnlp/twitter-roberta-base-sentiment-latest"
)

config = robertamodel.config

# Extrahiere Classification Head Weights
classifier_weights = robertamodel.classifier.out_proj.weight.data  # [3, 768]
classifier_bias = robertamodel.classifier.out_proj.bias.data      # [3]

print(f"Label Mapping: {config.id2label}")
# → {0: 'negative', 1: 'neutral', 2: 'positive'}


parquet_file = pq.ParquetFile(file_path)
os.makedirs(output_dir, exist_ok=True)

print(f"\nKlassifiziere {parquet_file.metadata.num_rows:,} Tweets...")

for i in range(parquet_file.metadata.num_row_groups):
    batch_file = os.path.join(output_dir, f'sentiment_results_batch_{i}.parquet')
    
    if os.path.exists(batch_file):
        print(f"Batch {i+1}/{parquet_file.metadata.num_row_groups}: Übersprungen")
        continue
    
    table = parquet_file.read_row_group(i, columns=['datetime', 'embedding'])
    df_chunk = table.to_pandas()
    
    # Embeddings → Logits (direkte Matrix-Multiplikation)
    embeddings = torch.tensor(np.vstack(df_chunk['embedding'].values), dtype=torch.float32)
    logits = embeddings @ classifier_weights.T + classifier_bias
    
    # Softmax → Predictions
    probs = torch.softmax(logits, dim=1)
    sentiment_ids = probs.argmax(dim=1).numpy()
    sentiments = [config.id2label[int(sid)] for sid in sentiment_ids]
    scores = probs.max(dim=1).values.numpy()
    
    df_chunk['sentiment'] = sentiments
    df_chunk['sentiment_score'] = scores
    
    df_chunk[['datetime', 'sentiment', 'sentiment_score']].to_parquet(batch_file, index=False)
    
    print(f"Batch {i+1}/{parquet_file.metadata.num_row_groups}: {len(df_chunk):,} Tweets ✓")
    
    del df_chunk, table, embeddings, logits, probs

print("\n✓ Fertig!")

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Label Mapping: {0: 'negative', 1: 'neutral', 2: 'positive'}

Klassifiziere 15,630,309 Tweets...
Batch 1/15: 1,048,576 Tweets ✓
Batch 2/15: 1,048,576 Tweets ✓
Batch 3/15: 1,048,576 Tweets ✓
Batch 4/15: 1,048,576 Tweets ✓
Batch 5/15: 1,048,576 Tweets ✓
Batch 6/15: 1,048,576 Tweets ✓
Batch 7/15: 1,048,576 Tweets ✓
Batch 8/15: 1,048,576 Tweets ✓
Batch 9/15: 1,048,576 Tweets ✓
Batch 10/15: 1,048,576 Tweets ✓
Batch 11/15: 1,048,576 Tweets ✓
Batch 12/15: 1,048,576 Tweets ✓
Batch 13/15: 1,048,576 Tweets ✓
Batch 14/15: 1,048,576 Tweets ✓
Batch 15/15: 950,245 Tweets ✓

✓ Fertig!


Aggregate the Embeddings pro Stunde i uploaded the results also

In [ ]:
print("Aggregiere Sentiment batch-weise...")
all_hourly = []

for i in range(15):
    batch_file = os.path.join(output_dir, f'sentiment_results_batch_{i}.parquet')
    
    df = pd.read_parquet(batch_file)
    df['datetime'] = pd.to_datetime(df['datetime'])
    df = df.set_index('datetime')
    
    # Aggregiere zu Stunden
    hourly = df.resample('1H').agg({
        'sentiment_score': 'mean',
        'sentiment': [
            lambda x: (x == 'positive').sum(),
            lambda x: (x == 'negative').sum(),
            'count'
        ]
    })
    
    hourly.columns = ['avg_score', 'n_positive', 'n_negative', 'total_tweets']
    all_hourly.append(hourly)
    
    print(f"Batch {i+1}/15 ✓")
    del df

# Kombiniere & speichere
sentiment_hourly = pd.concat(all_hourly).groupby(level=0).agg({
    'avg_score': 'mean',
    'n_positive': 'sum',
    'n_negative': 'sum',
    'total_tweets': 'sum'
})

sentiment_hourly['net_sentiment'] = sentiment_hourly['n_positive'] - sentiment_hourly['n_negative']

# WICHTIG: Entferne Timezone falls vorhanden
sentiment_hourly.index = sentiment_hourly.index.tz_localize(None)

sentiment_hourly.to_parquet('./data/sentiment_hourly_aggregated.parquet')
print(f"\n✓ Sentiment gespeichert: {len(sentiment_hourly):,} Stunden")

# BTC laden mit Unix Timestamp
btc = pd.read_csv(one_minute_data_dir)

# Konvertiere Unix Timestamp zu Datetime (ohne Timezone)
btc['datetime'] = pd.to_datetime(btc['Timestamp'], unit='s')
btc = btc.set_index('datetime')

print(f"\nBTC Daten: {len(btc):,} Minuten")
print(f"Zeitraum: {btc.index.min()} bis {btc.index.max()}")

# Aggregiere zu Stunden
btc_hourly = btc.resample('1H').agg({
    'Open': 'first',
    'High': 'max',
    'Low': 'min',
    'Close': 'last',
    'Volume': 'sum'
})

# Rename zu lowercase
btc_hourly.columns = [col.lower() for col in btc_hourly.columns]

print(f"BTC Hourly: {len(btc_hourly):,} Stunden")

# Check Timezones vor Merge
print(f"\nBTC Index Timezone: {btc_hourly.index.tz}")
print(f"Sentiment Index Timezone: {sentiment_hourly.index.tz}")

# Merge
merged = btc_hourly.join(sentiment_hourly, how='inner')
merged = merged.dropna()

print(f"\n" + "="*60)
print(f"✓ MERGED DATASET")
print("="*60)
print(f"Anzahl Stunden: {len(merged):,}")
print(f"Zeitraum: {merged.index.min()} bis {merged.index.max()}")
print(f"Spalten: {merged.columns.tolist()}")
print(f"\nErste Zeilen:")
print(merged.head())

# Speichere
merged.to_csv('./data/btc_sentiment_merged.csv')
print(f"\n✓ Gespeichert: btc_sentiment_merged.csv")

Stats and P-Tests for comprehension code should already work because i uploaded the important stuff in github

In [18]:
warnings.filterwarnings('ignore')

# Lade merged Dataset
merged = pd.read_csv('btc_sentiment_merged.csv', index_col=0, parse_dates=True)

print("="*60)
print("DATASET OVERVIEW")
print("="*60)
print(f"Zeitraum: {merged.index.min()} bis {merged.index.max()}")
print(f"Anzahl Stunden: {len(merged):,}")
print(f"Spalten: {merged.columns.tolist()}")
print(f"\nFehlende Werte:")
print(merged.isnull().sum())
print(f"\nErste Zeilen:")
print(merged.head())

# Berechne Returns und Features
merged['returns'] = merged['close'].pct_change()
merged['returns_1h'] = merged['returns'].shift(-1)  # Nächste Stunde (für Prediction)
merged['returns_24h'] = merged['close'].pct_change(24).shift(-24)  # 24h ahead
merged['price_direction'] = np.where(merged['returns_1h'] > 0, 1, -1)

# Log Returns (besser für Finanz-Analyse)
merged['log_returns'] = np.log(merged['close'] / merged['close'].shift(1))
merged['log_returns_1h'] = merged['log_returns'].shift(-1)

# Remove NaN
merged_clean = merged.dropna().copy()

print(f"\nNach Cleanup: {len(merged_clean):,} Stunden")

# =============================================================================
# ANALYSE 1: KORRELATIONEN
# =============================================================================
print("\n" + "="*60)
print("1. KORRELATIONS-ANALYSE")
print("="*60)

corr_features = ['returns_1h', 'avg_score', 'net_sentiment', 'total_tweets', 
                 'n_positive', 'n_negative']
corr_matrix = merged_clean[corr_features].corr()

print("\nKorrelation mit Returns (nächste Stunde):")
print(corr_matrix['returns_1h'].sort_values(ascending=False))

# Pearson Korrelation + P-Value


corr_sentiment_return, p_sentiment = pearsonr(
    merged_clean['avg_score'], 
    merged_clean['returns_1h']
)
corr_net_return, p_net = pearsonr(
    merged_clean['net_sentiment'], 
    merged_clean['returns_1h']
)

print(f"\nStatistische Tests:")
print(f"Avg Sentiment Score vs Returns: r={corr_sentiment_return:.4f}, p={p_sentiment:.4f}")
print(f"Net Sentiment vs Returns: r={corr_net_return:.4f}, p={p_net:.4f}")

if p_sentiment < 0.05:
    print("→ Sentiment-Score ist SIGNIFIKANT korreliert mit Returns! ✓")
else:
    print("→ Keine signifikante Korrelation gefunden.")

# =============================================================================
# ANALYSE 2: SENTIMENT-QUARTILE → RETURNS
# =============================================================================
print("\n" + "="*60)
print("2. SENTIMENT QUARTILE ANALYSE")
print("="*60)

# Teile Sentiment in Quartile
merged_clean['sentiment_quartile'] = pd.qcut(
    merged_clean['avg_score'], 
    q=4, 
    labels=['Q1_Bearish', 'Q2_Neutral_Low', 'Q3_Neutral_High', 'Q4_Bullish']
)

quartile_stats = merged_clean.groupby('sentiment_quartile', observed=True).agg({
    'returns_1h': ['mean', 'std', 'count'],
    'price_direction': lambda x: (x == 1).sum() / len(x)  # Win rate
})

quartile_stats.columns = ['avg_return_1h', 'std', 'n_samples', 'win_rate']
print(quartile_stats)

# T-Test: Bullish vs Bearish
bullish_returns = merged_clean[merged_clean['sentiment_quartile'] == 'Q4_Bullish']['returns_1h']
bearish_returns = merged_clean[merged_clean['sentiment_quartile'] == 'Q1_Bearish']['returns_1h']

t_stat, p_value = stats.ttest_ind(bullish_returns, bearish_returns)

print(f"\nT-Test (Bullish vs Bearish Sentiment):")
print(f"  Bullish Avg Return: {bullish_returns.mean()*100:.4f}%")
print(f"  Bearish Avg Return: {bearish_returns.mean()*100:.4f}%")
print(f"  Difference: {(bullish_returns.mean() - bearish_returns.mean())*100:.4f}%")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

if p_value < 0.05:
    print(f"  → SIGNIFIKANTER Unterschied! {'✓' if bullish_returns.mean() > bearish_returns.mean() else '⚠'}")
else:
    print("  → Kein signifikanter Unterschied")

# =============================================================================
# ANALYSE 3: GRANGER CAUSALITY (Führt Sentiment den Preis?)
# =============================================================================
print("\n" + "="*60)
print("3. GRANGER CAUSALITY TEST")
print("="*60)

# Normalisiere für Granger Test
data_granger = merged_clean[['returns', 'net_sentiment']].copy()
data_granger['net_sentiment_norm'] = (
    data_granger['net_sentiment'] - data_granger['net_sentiment'].mean()
) / data_granger['net_sentiment'].std()

max_lag = 12
print(f"Teste ob Sentiment die Returns der nächsten {max_lag} Stunden vorhersagt:\n")

try:
    granger_results = grangercausalitytests(
        data_granger[['returns', 'net_sentiment_norm']], 
        max_lag, 
        verbose=False
    )
    
    print("Lag  |  F-Test p-value  |  Signifikanz")
    print("-" * 45)
    for lag in range(1, max_lag + 1):
        p_value = granger_results[lag][0]['ssr_ftest'][1]
        sig = '***' if p_value < 0.01 else '**' if p_value < 0.05 else '*' if p_value < 0.1 else ''
        print(f"{lag:2d}h  |  {p_value:.6f}      |  {sig}")
    
    # Finde besten Lag
    best_lag = min(range(1, max_lag + 1), 
                   key=lambda l: granger_results[l][0]['ssr_ftest'][1])
    best_p = granger_results[best_lag][0]['ssr_ftest'][1]
    
    if best_p < 0.05:
        print(f"\n→ Sentiment hat PREDICTIVE POWER bei Lag {best_lag}h! (p={best_p:.4f}) ✓")
    else:
        print(f"\n→ Keine Granger Causality gefunden")
        
except Exception as e:
    print(f"Fehler beim Granger Test: {e}")

# =============================================================================
# ANALYSE 4: TRADING STRATEGY BACKTEST
# =============================================================================
print("\n" + "="*60)
print("4. TRADING STRATEGY BACKTEST")
print("="*60)

# Strategie 1: Long wenn Sentiment > Median
merged_clean['signal_simple'] = np.where(
    merged_clean['avg_score'] > merged_clean['avg_score'].median(), 
    1, -1
)

# Strategie 2: Long nur wenn Q4, Short nur wenn Q1, sonst neutral
merged_clean['signal_quartile'] = 0
merged_clean.loc[merged_clean['sentiment_quartile'] == 'Q4_Bullish', 'signal_quartile'] = 1
merged_clean.loc[merged_clean['sentiment_quartile'] == 'Q1_Bearish', 'signal_quartile'] = -1

# Berechne Strategy Returns
merged_clean['strat_simple'] = merged_clean['signal_simple'] * merged_clean['returns_1h']
merged_clean['strat_quartile'] = merged_clean['signal_quartile'] * merged_clean['returns_1h']

# Performance Metriken
def calc_performance(returns, name):
    total_return = (1 + returns).prod() - 1
    annualized_return = (1 + total_return) ** (365*24 / len(returns)) - 1
    sharpe = returns.mean() / returns.std() * np.sqrt(365*24) if returns.std() > 0 else 0
    max_drawdown = (returns.cumsum() - returns.cumsum().cummax()).min()
    win_rate = (returns > 0).sum() / len(returns)
    
    print(f"\n{name}:")
    print(f"  Total Return: {total_return*100:.2f}%")
    print(f"  Annualized Return: {annualized_return*100:.2f}%")
    print(f"  Sharpe Ratio: {sharpe:.3f}")
    print(f"  Max Drawdown: {max_drawdown*100:.2f}%")
    print(f"  Win Rate: {win_rate*100:.2f}%")
    
    return total_return

ret_bh = calc_performance(merged_clean['returns_1h'].dropna(), "Buy & Hold")
ret_simple = calc_performance(merged_clean['strat_simple'].dropna(), "Simple Strategy (Median)")
ret_quartile = calc_performance(merged_clean['strat_quartile'].dropna(), "Quartile Strategy")

print(f"\n{'='*60}")
print(f"PERFORMANCE VERGLEICH")
print(f"{'='*60}")
print(f"Buy & Hold:         {ret_bh*100:+.2f}%")
print(f"Simple Strategy:    {ret_simple*100:+.2f}% ({(ret_simple-ret_bh)*100:+.2f}%)")
print(f"Quartile Strategy:  {ret_quartile*100:+.2f}% ({(ret_quartile-ret_bh)*100:+.2f}%)")

# =============================================================================
# ANALYSE 5: EXTREME SENTIMENT EVENTS
# =============================================================================
print("\n" + "="*60)
print("5. EXTREME SENTIMENT EVENTS")
print("="*60)

# Top/Bottom 5%
top_5_threshold = merged_clean['avg_score'].quantile(0.95)
bottom_5_threshold = merged_clean['avg_score'].quantile(0.05)

extreme_bullish = merged_clean[merged_clean['avg_score'] > top_5_threshold]
extreme_bearish = merged_clean[merged_clean['avg_score'] < bottom_5_threshold]

print(f"\nExtrem Bullish (Top 5%): {len(extreme_bullish)} Events")
print(f"  Durchschnittlicher Return (nächste Stunde): {extreme_bullish['returns_1h'].mean()*100:.4f}%")
print(f"  Win Rate: {(extreme_bullish['returns_1h'] > 0).mean()*100:.2f}%")

print(f"\nExtrem Bearish (Bottom 5%): {len(extreme_bearish)} Events")
print(f"  Durchschnittlicher Return (nächste Stunde): {extreme_bearish['returns_1h'].mean()*100:.4f}%")
print(f"  Win Rate: {(extreme_bearish['returns_1h'] > 0).mean()*100:.2f}%")

# =============================================================================
# SAVE RESULTS
# =============================================================================
merged_clean.to_csv('./data/btc_sentiment_analysis_complete.csv')
print(f"\n{'='*60}")
print(f"✓ Vollständige Analyse gespeichert: btc_sentiment_analysis_complete.csv")
print(f"{'='*60}")


DATASET OVERVIEW
Zeitraum: 2021-01-01 00:00:00 bis 2021-12-31 23:00:00
Anzahl Stunden: 8,741
Spalten: ['open', 'high', 'low', 'close', 'volume', 'avg_score', 'n_positive', 'n_negative', 'total_tweets', 'net_sentiment']

Fehlende Werte:
open             0
high             0
low              0
close            0
volume           0
avg_score        0
n_positive       0
n_negative       0
total_tweets     0
net_sentiment    0
dtype: int64

Erste Zeilen:
                         open      high       low     close      volume  \
datetime                                                                  
2021-01-01 00:00:00  28912.47  29052.03  28776.17  28949.03  365.064315   
2021-01-01 01:00:00  28949.71  29200.00  28921.64  29198.34  520.772208   
2021-01-01 02:00:00  29200.00  29219.01  28947.70  28992.18  394.441884   
2021-01-01 03:00:00  28987.60  29191.60  28945.26  29144.56  220.722444   
2021-01-01 04:00:00  29135.25  29161.46  28843.74  28992.79  500.353472   

                    